# Coastal Flood EAD Analysis (Feb 2026)

This notebook reads coastal flood direct-damage outputs (with and without mangroves) for RP 25/100/500, ready for EAD calculations (sensitivity set6).


In [ ]:
from pathlib import Path
import sys
import importlib

import pandas
import numpy
import geopandas
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

pandas.set_option('display.max_columns', 200)
pandas.set_option('display.width', 200)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# Reporting in USD only
JMD_PER_USD = 150.0
USD_PER_JMD = 1.0 / JMD_PER_USD


In [ ]:
# Core paths
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
results_path = base_path / 'dphil_paper_3/results_coastal_set6'
direct_damages_path = results_path / 'direct_damages'
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'

if not direct_damages_path.exists():
    raise FileNotFoundError(f'Missing folder: {direct_damages_path}')
if not network_csv.exists():
    raise FileNotFoundError(f'Missing file: {network_csv}')

print(f'direct_damages_path: {direct_damages_path}')
print(f'network_csv: {network_csv}')


In [ ]:
# Discover all direct-damage parquet files produced for sensitivity parameter set 0
parquet_files = sorted(direct_damages_path.rglob('*_direct_damages_parameter_set_0.parquet'))

if not parquet_files:
    raise FileNotFoundError(f'No direct damage parquet files found under {direct_damages_path}')

print(f'Found {len(parquet_files)} direct-damage files.')
pandas.DataFrame({'parquet_file': [str(p) for p in parquet_files]})


In [ ]:
# Read network metadata and build expected file mapping (asset/layer/id column)
network_details = pandas.read_csv(network_csv)
required_cols = ['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']
missing_cols = [c for c in required_cols if c not in network_details.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in network csv: {missing_cols}')

network_details = network_details[required_cols].drop_duplicates().copy()
network_details['folder_name'] = network_details['asset_gpkg'] + '_' + network_details['asset_layer']
network_details['expected_parquet'] = network_details['folder_name'].apply(
    lambda folder: direct_damages_path / folder / f'{folder}_direct_damages_parameter_set_0.parquet'
)
network_details['exists'] = network_details['expected_parquet'].apply(lambda p: p.exists())

display(network_details[['sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column', 'exists']].sort_values(['sector', 'asset_gpkg', 'asset_layer']))

missing_files = network_details.loc[~network_details['exists'], ['asset_gpkg', 'asset_layer', 'expected_parquet']]
if len(missing_files) > 0:
    print('Missing expected files:')
    display(missing_files)
else:
    print('All expected direct-damage files are present.')


In [ ]:
# Load each damage table with the key columns needed for coastal EAD setup
required_damage_cols = [
    'coastal_flood_fn_mg_rp_25',
    'coastal_flood_fn_mg_rp_100',
    'coastal_flood_fn_mg_rp_500',
    'coastal_flood_fn_nomg_rp_25',
    'coastal_flood_fn_nomg_rp_100',
    'coastal_flood_fn_nomg_rp_500',
]

loaded_damage_tables = {}
load_summary_rows = []

for row in network_details.itertuples(index=False):
    parquet_path = row.expected_parquet
    if not parquet_path.exists():
        continue

    damage_df = pandas.read_parquet(parquet_path)

    missing = [c for c in required_damage_cols if c not in damage_df.columns]
    key = f'{row.asset_gpkg}_{row.asset_layer}'
    loaded_damage_tables[key] = damage_df

    load_summary_rows.append({
        'table_key': key,
        'sector': row.sector,
        'subsector': row.asset_description,
        'rows': len(damage_df),
        'columns': len(damage_df.columns),
        'missing_required_cols': ', '.join(missing) if missing else '',
        'parquet_path': str(parquet_path),
    })

load_summary = pandas.DataFrame(load_summary_rows).sort_values(['sector', 'table_key']).reset_index(drop=True)
display(load_summary)

bad_tables = load_summary[load_summary['missing_required_cols'] != '']
if len(bad_tables) > 0:
    print('Some tables are missing required coastal EAD columns:')
    display(bad_tables[['table_key', 'missing_required_cols']])
else:
    print('All loaded tables have the required RP 25/100/500 with/without mangrove columns.')


In [ ]:
# Optional preview: pick one loaded table
table_to_preview = 'roads_edges'  # change as needed

if table_to_preview not in loaded_damage_tables:
    print(f"'{table_to_preview}' not found. Available keys:")
    print(sorted(loaded_damage_tables.keys()))
else:
    preview_cols = [
        c for c in loaded_damage_tables[table_to_preview].columns
        if c.startswith('coastal_flood_fn_') and '_rp_' in c
    ]
    display(loaded_damage_tables[table_to_preview][preview_cols].head(10))


In [ ]:
# Import Robyn_river_floods library (without modifying the .py file)
robyn_lib_path = base_path / 'robyns_libraries'
if str(robyn_lib_path) not in sys.path:
    sys.path.append(str(robyn_lib_path))

import Robyn_river_floods
importlib.reload(Robyn_river_floods)

print(f'Using Robyn_river_floods from: {Robyn_river_floods.__file__}')


In [ ]:
# Compute asset-level EADs using RP25/100/500 for with-mangroves and without-mangroves (USD)
rp_mg_cols = [
    'coastal_flood_fn_mg_rp_25',
    'coastal_flood_fn_mg_rp_100',
    'coastal_flood_fn_mg_rp_500',
]
rp_nomg_cols = [
    'coastal_flood_fn_nomg_rp_25',
    'coastal_flood_fn_nomg_rp_100',
    'coastal_flood_fn_nomg_rp_500',
]

mg_to_rp = {
    'coastal_flood_fn_mg_rp_25': 'rp25.0',
    'coastal_flood_fn_mg_rp_100': 'rp100.0',
    'coastal_flood_fn_mg_rp_500': 'rp500.0',
}
nomg_to_rp = {
    'coastal_flood_fn_nomg_rp_25': 'rp25.0',
    'coastal_flood_fn_nomg_rp_100': 'rp100.0',
    'coastal_flood_fn_nomg_rp_500': 'rp500.0',
}

asset_ead_rows = []

for row in network_details.itertuples(index=False):
    table_key = f'{row.asset_gpkg}_{row.asset_layer}'
    if table_key not in loaded_damage_tables:
        continue

    damage_df = loaded_damage_tables[table_key].copy()

    needed = [row.asset_id_column] + rp_mg_cols + rp_nomg_cols
    missing = [c for c in needed if c not in damage_df.columns]
    if missing:
        print(f"Skipping {table_key}: missing columns {missing}")
        continue

    grouped = (
        damage_df[[row.asset_id_column] + rp_mg_cols + rp_nomg_cols]
        .groupby(row.asset_id_column, as_index=False)
        .sum()
        .copy()
    )

    mg_df = grouped[rp_mg_cols].rename(columns=mg_to_rp)
    nomg_df = grouped[rp_nomg_cols].rename(columns=nomg_to_rp)

    ead_with_mg_jmd = Robyn_river_floods.calculate_ead(mg_df)
    ead_without_mg_jmd = Robyn_river_floods.calculate_ead(nomg_df)

    grouped['EAD_With_Mangroves_USD'] = ead_with_mg_jmd * USD_PER_JMD
    grouped['EAD_Without_Mangroves_USD'] = ead_without_mg_jmd * USD_PER_JMD
    grouped['Avoided_EAD_USD'] = grouped['EAD_Without_Mangroves_USD'] - grouped['EAD_With_Mangroves_USD']
    grouped['Avoided_EAD_Share_of_Baseline'] = grouped['Avoided_EAD_USD'] / grouped['EAD_Without_Mangroves_USD']
    grouped.loc[grouped['EAD_Without_Mangroves_USD'] <= 0, 'Avoided_EAD_Share_of_Baseline'] = pandas.NA

    grouped['Sector'] = row.sector
    grouped['Subsector'] = row.asset_description
    grouped['Asset'] = row.asset_gpkg
    grouped['Layer'] = row.asset_layer
    grouped = grouped.rename(columns={row.asset_id_column: 'Asset_ID'})

    out_cols = [
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
        'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'Avoided_EAD_USD', 'Avoided_EAD_Share_of_Baseline'
    ]
    asset_ead_rows.append(grouped[out_cols])

asset_ead = pandas.concat(asset_ead_rows, ignore_index=True) if asset_ead_rows else pandas.DataFrame()

print(f'Asset rows with EAD (USD): {len(asset_ead):,}')
display(asset_ead.head(20))


In [ ]:
# Summaries: subsector and sector EAD (USD)
if asset_ead.empty:
    raise ValueError('asset_ead is empty; check missing columns or source files.')

subsector_ead_summary = (
    asset_ead
    .groupby(['Sector', 'Subsector'], as_index=False)[
        ['EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'Avoided_EAD_USD']
    ]
    .sum()
)
subsector_ead_summary['Avoided_EAD_Share_of_Baseline'] = (
    subsector_ead_summary['Avoided_EAD_USD'] / subsector_ead_summary['EAD_Without_Mangroves_USD']
)
subsector_ead_summary.loc[
    subsector_ead_summary['EAD_Without_Mangroves_USD'] <= 0,
    'Avoided_EAD_Share_of_Baseline'
] = pandas.NA

sector_ead_summary = (
    asset_ead
    .groupby(['Sector'], as_index=False)[
        ['EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'Avoided_EAD_USD']
    ]
    .sum()
)
sector_ead_summary['Avoided_EAD_Share_of_Baseline'] = (
    sector_ead_summary['Avoided_EAD_USD'] / sector_ead_summary['EAD_Without_Mangroves_USD']
)
sector_ead_summary.loc[
    sector_ead_summary['EAD_Without_Mangroves_USD'] <= 0,
    'Avoided_EAD_Share_of_Baseline'
] = pandas.NA

print('Subsector EAD summary (USD):')
display(subsector_ead_summary.sort_values(['Sector', 'Subsector']))

print('Sector EAD summary (USD):')
display(sector_ead_summary.sort_values(['Sector']))


In [ ]:
# Save outputs (USD only)
output_damage_estimates = results_path / 'damage_estimates'
output_damage_estimates.mkdir(parents=True, exist_ok=True)

asset_out = output_damage_estimates / 'coastal_ead_asset_level_usd.csv'
subsector_out = output_damage_estimates / 'coastal_ead_subsector_summary_usd.csv'
sector_out = output_damage_estimates / 'coastal_ead_sector_summary_usd.csv'

asset_ead.to_csv(asset_out, index=False)
subsector_ead_summary.to_csv(subsector_out, index=False)
sector_ead_summary.to_csv(sector_out, index=False)

print(f'Saved: {asset_out}')
print(f'Saved: {subsector_out}')
print(f'Saved: {sector_out}')


In [ ]:
# USD output preview
print('Asset-level EAD in USD:')
display(asset_ead.head(20))

print('Subsector EAD summary in USD:')
display(subsector_ead_summary.sort_values(['Sector', 'Subsector']))

print('Sector EAD summary in USD:')
display(sector_ead_summary.sort_values(['Sector']))


In [ ]:
# Percentage of avoided EAD relative to no-mangroves EAD (USD)
# Formula: % avoided = 100 * Avoided_EAD / EAD_Without_Mangroves
subsector_pct = subsector_ead_summary.copy()
sector_pct = sector_ead_summary.copy()

for df in [subsector_pct, sector_pct]:
    df['Percent_Avoided_EAD_vs_NoMangroves'] = pandas.NA
    valid = df['EAD_Without_Mangroves_USD'] > 0
    df.loc[valid, 'Percent_Avoided_EAD_vs_NoMangroves'] = (
        100.0 * df.loc[valid, 'Avoided_EAD_USD'] / df.loc[valid, 'EAD_Without_Mangroves_USD']
    )

print('Sector-level percentage avoided EAD (% of no-mangroves EAD):')
display(
    sector_pct[['Sector', 'EAD_Without_Mangroves_USD', 'Avoided_EAD_USD', 'Percent_Avoided_EAD_vs_NoMangroves']]
    .sort_values('Sector')
    .round({'Percent_Avoided_EAD_vs_NoMangroves': 2})
)

print('Subsector-level percentage avoided EAD (% of no-mangroves EAD):')
display(
    subsector_pct[['Sector', 'Subsector', 'EAD_Without_Mangroves_USD', 'Avoided_EAD_USD', 'Percent_Avoided_EAD_vs_NoMangroves']]
    .sort_values(['Sector', 'Subsector'])
    .round({'Percent_Avoided_EAD_vs_NoMangroves': 2})
)

out_dir = results_path / 'damage_estimates'
sector_pct_out = out_dir / 'coastal_ead_sector_summary_usd_with_pct_avoided.csv'
subsector_pct_out = out_dir / 'coastal_ead_subsector_summary_usd_with_pct_avoided.csv'

sector_pct.to_csv(sector_pct_out, index=False)
subsector_pct.to_csv(subsector_pct_out, index=False)

print(f'Saved: {sector_pct_out}')
print(f'Saved: {subsector_pct_out}')


In [ ]:
# Build geospatial layers for mapping sector avoided EAD using coastal split geometries (USD)
if 'asset_ead' not in globals():
    raise ValueError('Run EAD computation cells first so asset_ead exists in memory.')

network_map_details = network_details[[
    'sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column'
]].drop_duplicates().copy()

map_layers = []
missing_split_files = []

for row in network_map_details.itertuples(index=False):
    split_file = results_path / f"{row.asset_gpkg}_splits__coastal_flood_rasters_fixed_for_intersections__{row.asset_layer}.geoparquet"
    if not split_file.exists():
        missing_split_files.append(str(split_file))
        continue

    split_geom = geopandas.read_parquet(split_file)
    if split_geom.crs is not None:
        split_geom = split_geom.to_crs('EPSG:3448')
    if row.asset_id_column not in split_geom.columns:
        print(f"Skipping {row.asset_gpkg}_{row.asset_layer}: id column '{row.asset_id_column}' not in split file")
        continue

    split_geom = split_geom[[row.asset_id_column, 'geometry']].copy()
    split_geom = geopandas.GeoDataFrame(split_geom, geometry='geometry', crs=split_geom.crs)

    ead_subset = asset_ead.loc[
        (asset_ead['Asset'] == row.asset_gpkg) & (asset_ead['Layer'] == row.asset_layer),
        ['Asset_ID', 'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD']
    ].copy()

    if ead_subset.empty:
        continue

    split_geom['_join_id'] = split_geom[row.asset_id_column].astype(str)
    ead_subset['_join_id'] = ead_subset['Asset_ID'].astype(str)

    merged = split_geom.merge(
        ead_subset[['_join_id', 'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD']],
        on='_join_id',
        how='left'
    )

    merged['Sector'] = row.sector
    merged['Subsector'] = row.asset_description
    merged['Asset'] = row.asset_gpkg
    merged['Layer'] = row.asset_layer
    merged['Asset_ID'] = merged[row.asset_id_column]
    merged['Avoided_EAD_USD'] = merged['Avoided_EAD_USD'].fillna(0.0)

    map_layers.append(merged[[
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
        'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'geometry'
    ]])

if not map_layers:
    raise ValueError('No map layers could be built from split files.')

sector_avoided_ead_map_layers = geopandas.GeoDataFrame(
    pandas.concat(map_layers, ignore_index=True),
    geometry='geometry',
    crs='EPSG:3448'
)

print(f"Map features loaded: {len(sector_avoided_ead_map_layers):,}")
print('Features by sector:')
display(sector_avoided_ead_map_layers.groupby('Sector', as_index=False).size())

if missing_split_files:
    print('Missing split files (skipped):')
    for p in sorted(set(missing_split_files)):
        print('-', p)


In [ ]:
# Plot avoided EAD maps for each sector with a shared global scale (USD)

jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')

jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs('EPSG:3448')
map_out_dir = results_path / 'damage_estimates' / 'maps_avoided_ead'
map_out_dir.mkdir(parents=True, exist_ok=True)

sector_order = ['buildings', 'energy', 'transport', 'water']
value_col = 'Avoided_EAD_USD'

# Red-white-green palette: red (damage increase), white (zero), green (avoided)
cmap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)

# Shared global scale using robust percentile to avoid all-yellow maps
all_vals = sector_avoided_ead_map_layers[value_col].fillna(0.0)
abs_vals = all_vals.abs()
global_abs_max = float(abs_vals.quantile(0.995))
if global_abs_max <= 0:
    global_abs_max = float(abs_vals.max()) if float(abs_vals.max()) > 0 else 1.0

norm = TwoSlopeNorm(vmin=-global_abs_max, vcenter=0.0, vmax=global_abs_max)
print(f'Global shared USD scale (robust): {-global_abs_max:,.2f} to +{global_abs_max:,.2f} (center 0)')

for sector_name in sector_order:
    sector_gdf = sector_avoided_ead_map_layers[
        sector_avoided_ead_map_layers['Sector'] == sector_name
    ].copy()

    if sector_gdf.empty:
        print(f'Skipping {sector_name}: no features')
        continue

    # Clip for plotting so outliers do not flatten color contrast
    sector_gdf['_plot_val'] = sector_gdf[value_col].clip(-global_abs_max, global_abs_max)

    fig, ax = plt.subplots(figsize=(10.5, 9.0))
    ax.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

    geom_type = sector_gdf.geometry.geom_type.astype(str)
    polys = sector_gdf[geom_type.str.contains('Polygon', na=False)]
    lines = sector_gdf[geom_type.str.contains('LineString', na=False)]
    points = sector_gdf[geom_type.str.contains('Point', na=False)]

    if not polys.empty:
        polys.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.12, edgecolor='none', alpha=0.9, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.9, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, markersize=16, alpha=0.95, zorder=4)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label('Avoided EAD (USD) | Green = avoided, Red = increase', rotation=90)

    ax.set_title(f"Avoided EAD map - {sector_name.capitalize()} (USD, shared scale)", fontsize=13)
    ax.set_axis_off()
    plt.tight_layout()

    out_png = map_out_dir / f"avoided_ead_map_{sector_name}_usd_shared_scale.png"
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    print(f'Saved: {out_png}')
    plt.show()


In [ ]:
# Plot avoided EAD maps for each sector with sector-specific accurate scales (USD)

jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')

jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs('EPSG:3448')
map_out_dir = results_path / 'damage_estimates' / 'maps_avoided_ead'
map_out_dir.mkdir(parents=True, exist_ok=True)

sector_order = ['buildings', 'energy', 'transport', 'water']
value_col = 'Avoided_EAD_USD'

# Red = increased damage (negative avoided), White = no change, Green = avoided damage
cmap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)

for sector_name in sector_order:
    sector_gdf = sector_avoided_ead_map_layers[
        sector_avoided_ead_map_layers['Sector'] == sector_name
    ].copy()

    if sector_gdf.empty:
        print(f'Skipping {sector_name}: no features')
        continue

    vals = sector_gdf[value_col].fillna(0.0)
    sector_abs_max = float(vals.abs().max())
    if sector_abs_max <= 0:
        sector_abs_max = 1.0

    norm = TwoSlopeNorm(vmin=-sector_abs_max, vcenter=0.0, vmax=sector_abs_max)
    print(f"{sector_name.capitalize()} scale (USD): {-sector_abs_max:,.2f} to +{sector_abs_max:,.2f}")

    fig, ax = plt.subplots(figsize=(10.5, 9.0))
    ax.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=ax, color='#c7c7c7', linewidth=0.4, zorder=1)

    geom_type = sector_gdf.geometry.geom_type.astype(str)
    polys = sector_gdf[geom_type.str.contains('Polygon', na=False)]
    lines = sector_gdf[geom_type.str.contains('LineString', na=False)]
    points = sector_gdf[geom_type.str.contains('Point', na=False)]

    if not polys.empty:
        polys.plot(ax=ax, column=value_col, cmap=cmap, norm=norm, linewidth=0.1, edgecolor='none', alpha=0.92, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column=value_col, cmap=cmap, norm=norm, linewidth=1.0, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column=value_col, cmap=cmap, norm=norm, markersize=18, alpha=0.95, zorder=4)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label('Avoided EAD (USD) | Red = increase, White = no change, Green = avoided', rotation=90)

    ax.set_title(f"Avoided EAD map - {sector_name.capitalize()} (USD, sector-specific scale)", fontsize=13)
    ax.set_axis_off()
    plt.tight_layout()

    out_png = map_out_dir / f"avoided_ead_map_{sector_name}_usd_sector_scale.png"
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    print(f'Saved: {out_png}')
    plt.show()


In [ ]:
# Plot USD avoided EAD maps with percentile clipping + explicit outlier highlighting

jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')

jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs('EPSG:3448')
map_out_dir = results_path / 'damage_estimates' / 'maps_avoided_ead'
map_out_dir.mkdir(parents=True, exist_ok=True)

sector_order = ['buildings', 'energy', 'transport', 'water']
value_col = 'Avoided_EAD_USD'
display_quantile = 0.995
outlier_highlight_color = '#145a32'

# Red = increased damage (negative avoided), White = no change, Green = avoided damage
cmap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)

for sector_name in sector_order:
    sector_gdf = sector_avoided_ead_map_layers[
        sector_avoided_ead_map_layers['Sector'] == sector_name
    ].copy()

    if sector_gdf.empty:
        print(f'Skipping {sector_name}: no features')
        continue

    vals = sector_gdf[value_col].fillna(0.0)
    abs_vals = vals.abs()

    sector_abs_max = float(abs_vals.max())
    display_cap = float(abs_vals.quantile(display_quantile))

    if display_cap <= 0:
        display_cap = sector_abs_max if sector_abs_max > 0 else 1.0
    if sector_abs_max <= 0:
        sector_abs_max = display_cap

    sector_gdf['_plot_val'] = vals.clip(-display_cap, display_cap)
    sector_gdf['_is_outlier'] = abs_vals > display_cap

    norm = TwoSlopeNorm(vmin=-display_cap, vcenter=0.0, vmax=display_cap)

    n_outliers = int(sector_gdf['_is_outlier'].sum())
    print(
        f"{sector_name.capitalize()}: display scale +/-{display_cap:,.2f} USD "
        f"(q={display_quantile:.3f}), true max abs={sector_abs_max:,.2f} USD, outliers={n_outliers}"
    )

    fig, ax = plt.subplots(figsize=(10.5, 9.0))
    ax.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=ax, color='#c7c7c7', linewidth=0.35, zorder=1)

    geom_type = sector_gdf.geometry.geom_type.astype(str)
    polys = sector_gdf[geom_type.str.contains('Polygon', na=False)]
    lines = sector_gdf[geom_type.str.contains('LineString', na=False)]
    points = sector_gdf[geom_type.str.contains('Point', na=False)]

    if not polys.empty:
        polys.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.08, edgecolor='none', alpha=0.92, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.95, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, markersize=17, alpha=0.95, zorder=4)

    outliers = sector_gdf[sector_gdf['_is_outlier']].copy()
    if not outliers.empty:
        out_geom_type = outliers.geometry.geom_type.astype(str)
        out_polys = outliers[out_geom_type.str.contains('Polygon', na=False)]
        out_lines = outliers[out_geom_type.str.contains('LineString', na=False)]
        out_points = outliers[out_geom_type.str.contains('Point', na=False)]

        if not out_polys.empty:
            out_polys.boundary.plot(ax=ax, color=outlier_highlight_color, linewidth=1.2, alpha=0.95, zorder=5)
        if not out_lines.empty:
            out_lines.plot(ax=ax, color=outlier_highlight_color, linewidth=2.1, alpha=0.95, zorder=5)
        if not out_points.empty:
            out_points.plot(ax=ax, color=outlier_highlight_color, markersize=36, alpha=0.95, zorder=5)

        # Label the largest outliers so they are easy to find in-map
        top_outliers = outliers.assign(_abs_val=outliers[value_col].abs()).sort_values('_abs_val', ascending=False).head(8).copy()
        label_points = top_outliers.geometry.representative_point()
        for (_, row), pt in zip(top_outliers.iterrows(), label_points):
            ax.text(
                pt.x,
                pt.y,
                f"{row[value_col]:,.0f}",
                fontsize=7,
                color=outlier_highlight_color,
                ha='left',
                va='bottom',
                zorder=6,
                bbox={'facecolor': 'white', 'alpha': 0.75, 'edgecolor': outlier_highlight_color, 'pad': 0.4}
            )

        # Save a table of outliers for auditing / finding exact assets
        outlier_table = outliers.assign(_abs_val=outliers[value_col].abs()).sort_values('_abs_val', ascending=False).copy()
        rp = outlier_table.geometry.representative_point()
        outlier_table['label_x'] = rp.x
        outlier_table['label_y'] = rp.y
        keep_cols = [
            'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', value_col,
            'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'label_x', 'label_y'
        ]
        outlier_csv = map_out_dir / f"avoided_ead_outliers_{sector_name}_usd_q{int(display_quantile * 1000)}.csv"
        outlier_table[keep_cols].to_csv(outlier_csv, index=False)
        print(f'Saved outlier table: {outlier_csv}')

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label(
        f"Avoided EAD (USD), clipped at q={display_quantile:.3f} | Red=increase, White=no change, Green=avoided",
        rotation=90,
    )

    ax.set_title(
        f"Avoided EAD map - {sector_name.capitalize()} (USD, clipped display + outlier highlights)",
        fontsize=12,
    )
    ax.set_axis_off()
    plt.tight_layout()

    out_png = map_out_dir / f"avoided_ead_map_{sector_name}_usd_q{int(display_quantile * 1000)}_outliers.png"
    fig.savefig(out_png, dpi=300, bbox_inches='tight')
    print(f'Saved map: {out_png}')
    plt.show()


In [ ]:
# Heuristic attribution: allocate avoided EAD (USD) to nearby Forces-of-Nature mangroves

if 'sector_avoided_ead_map_layers' not in globals():
    raise ValueError('Run the map-layer build cell first so sector_avoided_ead_map_layers exists.')

mangrove_path = base_path / 'dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'
if not mangrove_path.exists():
    raise FileNotFoundError(f'Missing mangrove file: {mangrove_path}')

mangrove_buffer_m = 1000  # Change to 500 or 250 for a tighter "local area" definition

# Build one geometry per asset to avoid double-counting split geometries
asset_key_cols = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']
asset_gdf = sector_avoided_ead_map_layers[asset_key_cols + ['Avoided_EAD_USD', 'geometry']].copy()
asset_gdf['Asset_ID'] = asset_gdf['Asset_ID'].astype(str)
asset_gdf = asset_gdf.dissolve(
    by=asset_key_cols,
    as_index=False,
    aggfunc={'Avoided_EAD_USD': 'first'}
)
asset_gdf = geopandas.GeoDataFrame(asset_gdf, geometry='geometry', crs='EPSG:3448')

# Read FoN mangroves and create proximity buffers
mangroves = geopandas.read_file(mangrove_path).to_crs('EPSG:3448')
if 'ID' not in mangroves.columns:
    raise KeyError("Expected an 'ID' column in mangrove shapefile.")

mangroves['Mangrove_ID'] = mangroves['ID'].astype(int)
mangrove_base_cols = ['Mangrove_ID']
for c in ['Parish', 'HECTARES', 'TYPE']:
    if c in mangroves.columns:
        mangrove_base_cols.append(c)

mangrove_buffers = mangroves[mangrove_base_cols + ['geometry']].copy()
mangrove_buffers['geometry'] = mangrove_buffers.geometry.buffer(mangrove_buffer_m)

# Spatially join assets to nearby mangrove buffers
joined = geopandas.sjoin(
    asset_gdf,
    mangrove_buffers[['Mangrove_ID', 'geometry']],
    how='left',
    predicate='intersects',
)

joined['nearby_mangrove_count'] = joined.groupby(asset_key_cols)['Mangrove_ID'].transform(lambda s: s.notna().sum())
joined['nearby_mangrove_count'] = joined['nearby_mangrove_count'].fillna(0).astype(int)

# Equal split across nearby mangroves (heuristic attribution)
joined['Avoided_EAD_USD_attributed'] = numpy.where(
    (joined['Mangrove_ID'].notna()) & (joined['nearby_mangrove_count'] > 0),
    joined['Avoided_EAD_USD'] / joined['nearby_mangrove_count'],
    0.0,
)

# Totals and diagnostics
asset_unique = asset_gdf[asset_key_cols + ['Avoided_EAD_USD']].copy()
total_avoided_usd = float(asset_unique['Avoided_EAD_USD'].sum())

matched_assets = joined.loc[joined['nearby_mangrove_count'] > 0, asset_key_cols].drop_duplicates()
matched_assets['matched'] = 1
asset_match_status = asset_unique.merge(matched_assets, on=asset_key_cols, how='left')
asset_match_status['matched'] = asset_match_status['matched'].fillna(0).astype(int)

attributed_total_usd = float(joined['Avoided_EAD_USD_attributed'].sum())
unattributed_total_usd = float(asset_match_status.loc[asset_match_status['matched'] == 0, 'Avoided_EAD_USD'].sum())

# Mangrove-level attribution tables
joined_m = joined.dropna(subset=['Mangrove_ID']).copy()
joined_m['Mangrove_ID'] = joined_m['Mangrove_ID'].astype(int)

mangrove_sector_summary = (
    joined_m.groupby(['Mangrove_ID', 'Sector'], as_index=False)['Avoided_EAD_USD_attributed']
    .sum()
    .sort_values(['Mangrove_ID', 'Sector'])
)

mangrove_total_summary = (
    joined_m.groupby('Mangrove_ID', as_index=False)['Avoided_EAD_USD_attributed']
    .sum()
    .rename(columns={'Avoided_EAD_USD_attributed': 'Total_Avoided_EAD_USD_attributed'})
    .sort_values('Total_Avoided_EAD_USD_attributed', ascending=False)
)

mangrove_attribution_map = mangroves[mangrove_base_cols + ['geometry']].merge(
    mangrove_total_summary,
    on='Mangrove_ID',
    how='left'
)
mangrove_attribution_map['Total_Avoided_EAD_USD_attributed'] = mangrove_attribution_map['Total_Avoided_EAD_USD_attributed'].fillna(0.0)

# Save outputs
out_dir = results_path / 'damage_estimates' / 'mangrove_attribution'
out_dir.mkdir(parents=True, exist_ok=True)

joined_out = joined[[
    'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', 'Avoided_EAD_USD',
    'Mangrove_ID', 'nearby_mangrove_count', 'Avoided_EAD_USD_attributed'
]].copy()
joined_out.to_csv(out_dir / f'asset_to_nearby_mangrove_attribution_{mangrove_buffer_m}m.csv', index=False)
mangrove_sector_summary.to_csv(out_dir / f'mangrove_attribution_by_sector_{mangrove_buffer_m}m.csv', index=False)
mangrove_total_summary.to_csv(out_dir / f'mangrove_attribution_total_{mangrove_buffer_m}m.csv', index=False)
mangrove_attribution_map.to_file(out_dir / f'mangrove_attribution_total_{mangrove_buffer_m}m.gpkg', driver='GPKG')

print(f'Attribution buffer: {mangrove_buffer_m} m')
print(f'Total avoided EAD (USD) across assets: {total_avoided_usd:,.2f}')
print(f'Total attributed to nearby mangroves (USD): {attributed_total_usd:,.2f}')
print(f'Total not attributed (outside buffer) (USD): {unattributed_total_usd:,.2f}')
if abs(total_avoided_usd) > 0:
    print(f'Fraction attributed: {100 * attributed_total_usd / total_avoided_usd:,.1f}%')
else:
    print('Fraction attributed: not defined (total avoided EAD is zero).')

print('Top 10 mangroves by attributed avoided EAD (USD):')
display(mangrove_total_summary.head(10))

print('Most negative 10 mangroves (associated with increased damage) (USD):')
display(mangrove_total_summary.sort_values('Total_Avoided_EAD_USD_attributed').head(10))


In [ ]:
# Map mangrove-level attributed avoided EAD (USD)

# Reuse in-memory output from the attribution cell, or load latest saved file
if 'out_dir' not in globals():
    out_dir = results_path / 'damage_estimates' / 'mangrove_attribution'
out_dir.mkdir(parents=True, exist_ok=True)

if 'mangrove_attribution_map' not in globals():
    gpkg_candidates = sorted(out_dir.glob('mangrove_attribution_total_*m.gpkg'))
    if not gpkg_candidates:
        raise FileNotFoundError('No mangrove attribution GPKG found. Run the attribution cell first.')
    latest_gpkg = gpkg_candidates[-1]
    mangrove_attribution_map = geopandas.read_file(latest_gpkg)
    print(f'Loaded: {latest_gpkg}')

if mangrove_attribution_map.crs is None or str(mangrove_attribution_map.crs).upper() != 'EPSG:3448':
    mangrove_attribution_map = mangrove_attribution_map.to_crs('EPSG:3448')

value_col = 'Total_Avoided_EAD_USD_attributed'
if value_col not in mangrove_attribution_map.columns:
    raise KeyError(f"Column '{value_col}' not found in mangrove attribution map data.")

jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')
jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs('EPSG:3448')

vals = mangrove_attribution_map[value_col].fillna(0.0)
abs_vals = vals.abs()
true_max_abs = float(abs_vals.max())

# Clipped display for readability in case a few mangroves dominate the scale
display_quantile = 0.995
display_cap = float(abs_vals.quantile(display_quantile))
if display_cap <= 0:
    display_cap = true_max_abs if true_max_abs > 0 else 1.0

mangrove_plot = mangrove_attribution_map.copy()
mangrove_plot['_plot_val'] = vals.clip(-display_cap, display_cap)

cmap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)
norm = TwoSlopeNorm(vmin=-display_cap, vcenter=0.0, vmax=display_cap)

fig, ax = plt.subplots(figsize=(10.5, 9.0))
ax.set_facecolor('#ffffff')
jamaica_boundary.boundary.plot(ax=ax, color='#c7c7c7', linewidth=0.35, zorder=1)

mangrove_plot.plot(
    ax=ax,
    column='_plot_val',
    cmap=cmap,
    norm=norm,
    linewidth=0.25,
    edgecolor='#6f6f6f',
    alpha=0.95,
    zorder=2,
)

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
cbar.set_label(
    f"Attributed avoided EAD (USD), clipped at q={display_quantile:.3f} | Red=increase, White=no change, Green=avoided",
    rotation=90,
)

ax.set_title(
    f"Forces-of-Nature mangroves: attributed avoided EAD (USD) | true max abs={true_max_abs:,.2f}",
    fontsize=12,
)
ax.set_axis_off()
plt.tight_layout()

buffer_label = str(mangrove_buffer_m) if 'mangrove_buffer_m' in globals() else 'latest'
out_png = out_dir / f"mangrove_attribution_map_usd_{buffer_label}m_q{int(display_quantile * 1000)}.png"
fig.savefig(out_png, dpi=300, bbox_inches='tight')
print(f'Saved: {out_png}')
plt.show()
